# Content-based Filtering

Run `train_test_split_data.ipynb` first. This notebook uses only prepared local
files. It evaluates this algorithm in its own scenario, not against another model.
This evaluates users with positive content profiles. Items with available metadata need not have past interactions. Snapshot text/category/free flags are assumed available at each historical boundary.

The order is load -> fit train -> validate -> select settings -> refit train plus
validation -> final test -> save. Run the final testing and saving cells only
after selecting settings. Fitting and scoring use local data. The final publication
cell uploads the saved model to Supabase when run deliberately.


In [4]:
import sys
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "config" / "ml_config.py").is_file():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config.ml_config import MODEL_ARTEFACT_DIR, DATA_OUTPUT_DIR
from data_loader import DataLoader
from models_loader import save_model
from src.content_based_filtering import model, recommender, tool_config

CANDIDATES = [{
    "max_features": tool_config.TFIDF_MAX_FEATURES,
    "ngram_range": tool_config.TFIDF_NGRAM_RANGE,
    "sublinear_tf": tool_config.TFIDF_SUBLINEAR_TF,
}]


### Load training, validation, and test inputs

Training and validation preparation use train history. Test preparation uses train + validation history. All ratings remain available for seen-item exclusion; latest preferences and recency are calculated only from each phase's past. Targets are first encounters with unseen products rated >=4. Unsupported targets and users are counted separately. Loading test data does not score a model.

In [5]:
data = DataLoader(output_dir=DATA_OUTPUT_DIR)
training = data.for_content_based(phase="train")
validation = data.for_content_based(phase="validation")
test = data.for_content_based(phase="test")

prepared = {"train": training, "validation": validation, "test": test}

In [6]:
pd.DataFrame([
    {
        "phase": name,
        "items": len(phase.items),
        "history_rows": len(phase.history),
        "history_users": phase.history.user_id.nunique(),
        "history_start": phase.history.review_date.min(),
        "history_end": phase.history.review_date.max(),
        "targets": len(phase.targets),
        "target_users": phase.targets.user_id.nunique(),
        "excluded_targets": phase.excluded_targets,
        "excluded_users": phase.excluded_users,
    } for name, phase in prepared.items()
])

,phase,items,history_rows,history_users,history_start,history_end,targets,target_users,excluded_targets,excluded_users
0,train,27797,194603,76552,2000-01-24 00:00:00+00:00,2019-12-06 00:00:00+00:00,0,0,0,0
1,validation,27797,194603,76552,2000-01-24 00:00:00+00:00,2019-12-06 00:00:00+00:00,6413,3349,9791,5884
2,test,27797,218838,84714,2000-01-24 00:00:00+00:00,2021-01-12 00:00:00+00:00,6258,3271,8970,5364


In [7]:
training.items.head()

,parent_asin,item_title,description,features,main_category,is_free
0,0071480935,"sing, watch, & learn spanish (dvd + guide): 16...",Latin rhythms that will get your kids singing ...,,software,False
1,0077431618,connect math hosted by aleks access card 52 we...,"About the Author,Sherri Messersmith has been t...",,software,False
2,0321700945,adobe photoshop lightroom 3 learn by video,"Product Description,This complete training pro...","New,Mint Condition,Dispatch same day for order...",software,False
3,0321719816,learn adobe dreamweaver cs5 by video: core tra...,"Product Description,This complete training pro...",,software,False
4,0321719824,adobe flash professional cs5: learn by video: ...,This complete training program from Adobe Pres...,,software,False


In [8]:
training.history.head()

,review_id,user_id,parent_asin,review_date,review_rating,recency_weight
0,241327,ae223tlcv274eett6yjjoj4imcza,b005k17ru0,2012-01-09 00:00:00+00:00,5,0.004159
1,241326,ae223tlcv274eett6yjjoj4imcza,b0092uelr2,2013-08-15 00:00:00+00:00,5,0.012597
2,241325,ae223tlcv274eett6yjjoj4imcza,b00irn4k2q,2014-03-30 00:00:00+00:00,4,0.019380
3,8438,ae224gvo7ohtyf26u6er6beviuaq,b00xrzy5zg,2016-07-04 00:00:00+00:00,5,0.093102
4,44713,ae22567o4nhejdeaxmkgeha24bwa,b071rlfy84,2016-03-11 00:00:00+00:00,5,0.074847


In [9]:
validation.targets.head()

,review_id,asin,parent_asin,user_id,review_date,review_title,review_text,review_images,verified_purchase,helpful_vote,...,review_title_length,review_word_count,is_extreme_rating,is_positive,is_negative,num_review_img,days_since_review,review_year,review_month,recency_weight
0,231770,b07t771sph,b07t771sph,aerb5hlpymtl6hqhdexyxzsi6w3a,2019-12-07 00:00:00+00:00,wonderful,i like the variety i get with you tube.,[],True,0,...,9,9,False,True,False,0,1200,2019,12,0.037425
1,4322,b00gfgp5hq,b00gfgp5hq,ahjjv4i5wkcvhf2cevb7fcnrt75a,2019-12-07 00:00:00+00:00,good challenge,i enjoy trying to do in shortest time,[],True,0,...,14,8,False,True,False,0,1200,2019,12,0.037425
2,142333,b00dvkoybm,b00dvkoybm,aetijb5yzilrtgeqhiqnsntm2hma,2019-12-07 00:00:00+00:00,like real pool,this is a fun game for its competitive format....,[],True,8,...,14,15,True,True,False,0,1200,2019,12,0.037425
3,198195,b019cvakmk,b019cvakmk,afyqh7qvcv6bvhy5eqt7nnummeza,2019-12-07 00:00:00+00:00,good,good addition to the channel.,[],True,0,...,4,5,True,True,False,0,1200,2019,12,0.037425
4,243015,b0066tuxu6,b0066tuxu6,afawza6mseqwrmnz673owkcqrzta,2019-12-07 00:00:00+00:00,good price,i got it for a good price. i love the original...,[],True,0,...,10,12,True,True,False,0,1200,2019,12,0.037425


### Build models on training history and validate each configuration

Fitting and evaluation are functions in src. Each candidate sees only training history; validation targets are used solely for model selection. Recall@K measures retrieval and NDCG@K measures ranking. They are macro-averaged over supported users with relevant targets. Empty returned lists count as misses; no target population means no score. No model files are written by this step.

In [10]:
if validation.targets.empty:
    raise ValueError("No supported validation targets. Review data coverage before fitting.")

validation_rows = []
best_score = None
selected_parameters = None
selected_bundle = None
selected_validation = None

for parameters in CANDIDATES:
    bundle = model.fit(
        training.items,
        **parameters
    )

    result = recommender.evaluate(
        bundle,
        validation,
        k=tool_config.DEFAULT_TOP_N
    )

    validation_rows.append({
        **parameters,
        **result["summary"]
    })

    score = (result["summary"]["ndcg"], result["summary"]["recall"])

    if best_score is None or score > best_score:
        best_score = score
        selected_parameters = dict(parameters)
        selected_bundle = bundle
        selected_validation = result

pd.DataFrame(validation_rows)

100%|██████████| 3349/3349 [33:40<00:00,  1.66it/s]


,max_features,ngram_range,sublinear_tf,k,users,recall,ndcg,excluded_targets,excluded_users
0,10000,"(1, 2)",True,10,3349,0.030533,0.017331,9791,5884


### Review the selected configuration

The highest validation NDCG is selected, with Recall as a tie-breaker. Revise CANDIDATES and rerun validation if needed. Freeze these settings before final test.

In [11]:
pd.DataFrame([selected_parameters])

,max_features,ngram_range,sublinear_tf
0,10000,"(1, 2)",True


In [12]:
pd.DataFrame([selected_validation["summary"]])

,k,users,recall,ndcg,excluded_targets,excluded_users
0,10,3349,0.030533,0.017331,9791,5884


In [13]:
selected_validation["per_user"].head()

,user_id,targets,returned,recall,ndcg
0,ae233jb2te3mwx2dxwknmepo4nba,1,10,0.0,0.0
1,ae23y76mksyhdrfcym65iytgixsa,1,10,0.0,0.0
2,ae24aljoiqjoibv3ycrbis4ekx2a,1,10,0.0,0.0
3,ae24ffsuqhe3j6nybicb7v2whuaa,2,10,0.0,0.0
4,ae2545unsqub2x3nibcs5upep5gq,1,10,0.0,0.0


### Refit and run final test deliberately

This fits using train + validation history and evaluates the untouched test targets. Do not tune settings based on the final score. The item representations are rebuilt for the test catalogue, and user profiles use only test.history for preferences and seen-item exclusion.

In [14]:
final_bundle = None
final_test = None

if test.targets.empty:
    raise ValueError("No supported test targets; there is no final ranking score.")

final_bundle = model.fit(
    test.items,
    **selected_parameters
)

final_test = recommender.evaluate(
    final_bundle,
    test,
    k=tool_config.DEFAULT_TOP_N
)

pd.DataFrame([final_test["summary"]])

100%|██████████| 3271/3271 [32:04<00:00,  1.70it/s]


,k,users,recall,ndcg,excluded_targets,excluded_users
0,10,3271,0.042049,0.023968,8970,5364


In [15]:
final_test["per_user"].head()

,user_id,targets,returned,recall,ndcg
0,ae23ldqtb7l76ap6e6wpbfvyl5da,2,10,0.0,0.0
1,ae2aiax4j4qhrwhsqbxisk4epila,1,10,0.0,0.0
2,ae2anh2xhvewkokb6iv7dvclde4a,1,10,0.0,0.0
3,ae2b2dy6zjvf7kz2kg24in7bhf6a,2,10,0.0,0.0
4,ae2byrptnrs5v57i5r25g3dwiseq,2,10,0.0,0.0


### Save the tested model

This saves one model.joblib and does not upload it. Run the next publishing cell deliberately to upload that saved bundle; see TRAINING.md.

In [16]:
if final_bundle is None or final_test is None:
    raise ValueError("Complete the final test before saving this serving model.")

model_path = MODEL_ARTEFACT_DIR / tool_config.MODEL_NAME / tool_config.MODEL_FILENAME
save_model(final_bundle, model_path)

print(f"Model Saved: {model_path}")

Model Saved: C:\Users\user\OneDrive\Desktop\Amozon\amozon_backend\models\content_based_filtering\model.joblib


### Push Saved Model into Supabase

Run this cell only after saving the tested model. It validates the saved bundle,
generates a model ID locally, uploads to the configured ModelArtefacts bucket,
then inserts its ModelRegistry row. Every successful run publishes a new ID.
An upload failure stops registration; a registry failure can leave an unregistered
uploaded file. No previous models are deleted.

Copy the printed ID into config/api_config.py MODEL_IDS before deployment.
Existing local model files take precedence; changing the ID does not refresh them.

In [17]:
import uuid
from config.ml_config import MODELREGISTRY_TABLE_NAME
from models_loader import load_model, validate_model

# Validate the saved bundle before making any remote write.
model_path = MODEL_ARTEFACT_DIR / tool_config.MODEL_NAME / tool_config.MODEL_FILENAME
validate_model(load_model(model_path), tool_config.MODEL_NAME)

from utils.supabase_utils import SUPABASE_CLIENT, ARTEFACTS_STORAGE

published_model_id = str(uuid.uuid4())
storage_path = f"{tool_config.MODEL_NAME}/{published_model_id}"

with model_path.open("rb") as model_file:
    ARTEFACTS_STORAGE.upload(
        path=f"{storage_path}/{tool_config.MODEL_FILENAME}",
        file=model_file,
    )

# Advertise the model only after its upload succeeds.
SUPABASE_CLIENT.table(MODELREGISTRY_TABLE_NAME).insert({
    "model_id": published_model_id,
    "tool": tool_config.MODEL_NAME,
    "storage_path": storage_path,
}).execute()

print(f'Update config/api_config.py: MODEL_IDS["{tool_config.MODEL_NAME}"] = "{published_model_id}"')

Update config/api_config.py: MODEL_IDS["content_based_filtering"] = "1fa76797-5438-4271-8aa0-0b74d64df9d6"
